# 12 - LOH.1 validation against the Prose reference

The **SCEC LOH.1** benchmark is a point double-couple buried at 2 km in a
low-velocity layer over a half-space, recorded at a surface receiver at
`(6, 8, 0)` km. The file `data/LOH.1_prose3` is the **semi-analytical reference
solution** (Prose), the de-facto truth for this problem.

The reference has 4 columns: `time`, **vertical**, **radial**, **transverse**
(the vertical column carries a `-1e5` scale, the horizontals `+1e5`). The raw
analytical traces are convolved with the LOH.1 source ramp
$(1/T^2)\,t\,e^{-t/T}$ ($T=0.1$ s) and a Gaussian ($\sigma=0.06$ s) to obtain
the seismogram.

ShakerMaker outputs `(Z, E, N)`; we rotate the horizontals into
**radial / transverse** along the source-receiver azimuth and compare. Each
figure is saved as a PNG in this folder.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from shakermaker import shakermaker
from shakermaker.cm_library.LOH import SCEC_LOH_1
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.stf_extensions.gaussian import Gaussian
from shakermaker.tools.plotting import SourcePlot

## Build the LOH.1 model

SCEC LOH.1 crust + a single Gaussian double-couple at 2 km, one surface
receiver at (6, 8, 0) km.

In [ ]:
sigma = 0.06
t0 = 6 * sigma
M0 = 1e18 / 5e14 / 2

crust = SCEC_LOH_1()
src = PointSource([0, 0, 2.0], [0., 90., 0.],
                  stf=Gaussian(t0=t0, freq=1/sigma, M0=M0, derivative=False))
fault = FaultSource([src], metadata={"name": "LOH1_source"})
sta = Station([6.0, 8.0, 0.0], metadata={"name": "loh1"})
stations = StationList([sta], {})

dt, nfft, tb, dk, tmax = 0.025, 4096, 1000, 0.1, 4096 * 0.025

## Preview the model before running

Inspect the crust (layered column + velocity profile), the source geometry, and
its source time function (STF). Each figure is saved as a PNG here.

In [ ]:
crust.plot()
plt.gcf().savefig("crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("crust_velocity_profile.png", dpi=150, bbox_inches="tight")

for _s in fault:
    _s.stf.dt = dt
SourcePlot(fault, show=False).savefig("source_geometry.png", dpi=150, bbox_inches="tight")

_stf = fault.get_source_by_id(0).stf
fig_stf, ax_stf = plt.subplots(figsize=(6, 2.5))
ax_stf.plot(_stf.t, _stf.data, color="tab:red", lw=1.5)
ax_stf.set_xlabel("time (s)")
ax_stf.set_ylabel("amplitude")
ax_stf.set_title("Source time function (STF)")
fig_stf.tight_layout()
fig_stf.savefig("source_stf.png", dpi=150, bbox_inches="tight")

## Run ShakerMaker

A single receiver, so this is fast. `check_parameters` reports the derived FK
parameters before running.

In [ ]:
model = shakermaker.ShakerMaker(crust, fault, stations)
model.check_parameters(dt=dt, nfft=nfft, dk=dk, tb=tb, tmax=tmax)
model.run(dt=dt, nfft=nfft, tb=tb, dk=dk, tmax=tmax, smth=1)
z, e, n, t = sta.get_response()

## Build the analytical reference

Load `LOH.1_prose3`, scale the columns (vertical / radial / transverse) and
convolve with the LOH.1 source ramp and the Gaussian.

In [ ]:
sig, T = 0.06, 0.1
A = np.loadtxt(os.path.join("..", "data", "LOH.1_prose3"))
nt = A.shape[0]
tref = A[:, 0]
dtr = tref[1] - tref[0]
vert = A[:, 1] * (-1e5)
rad = A[:, 2] * (1e5)
trans = A[:, 3] * (1e5)

ramp = (1.0 / T**2) * tref * np.exp(-tref / T)
rad = dtr * np.convolve(rad, ramp, "full")[:nt]
trans = dtr * np.convolve(trans, ramp, "full")[:nt]
vert = dtr * np.convolve(vert, ramp, "full")[:nt]

tau = tref - 6 * sig
factor = 1 - (2 * T / sig**2) * tau - ((T / sig)**2) * (1 - (tau / sig)**2)
gauss = (1.0 / (np.sqrt(2 * np.pi) * sig)) * factor * np.exp(-0.5 * (tau / sig)**2)
rad = dtr * np.convolve(rad, gauss, "full")[:nt]
trans = dtr * np.convolve(trans, gauss, "full")[:nt]
vert = dtr * np.convolve(vert, gauss, "full")[:nt]

## Rotate ShakerMaker to R/T/V and overlay

Source (0,0) -> receiver (6 North, 8 East), so the radial unit vector is
`(rN, rE) = (0.6, 0.8)`. We rotate the horizontals, interpolate onto the
reference grid and report the correlation per component.

In [ ]:
rN, rE = 0.6, 0.8
sm_radial = rN * n + rE * e
sm_transv = -rE * n + rN * e
sm_vert = z


def corr(a, b):
    a = a - a.mean()
    b = b - b.mean()
    den = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / den) if den > 0 else 0.0


pairs = [("Radial", sm_radial, rad),
         ("Transverse", sm_transv, trans),
         ("Vertical", sm_vert, vert)]

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
for ax, (name, smc, refc) in zip(axes, pairs):
    smi = np.interp(tref, t, smc)
    cc = corr(refc, smi)
    ax.plot(tref, refc, "k-", lw=1.6, label=f"Prose {name}")
    ax.plot(tref, smi, "r-", lw=1.0, alpha=0.85,
            label=f"ShakerMaker {name} (corr={cc:.3f})")
    ax.set_ylabel("velocity")
    ax.set_xlim([0, 9])
    ax.grid(alpha=0.3)
    ax.legend(loc="upper right")
axes[0].set_title("LOH.1 validation: Prose reference vs ShakerMaker", fontweight="bold")
axes[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.gcf().savefig("LOH1_validation.png", dpi=150, bbox_inches="tight")